In [0]:
pip install Faker

In [0]:
# =============================================================
# NOTEBOOK:  04_setup_sql_source_data
# PURPOSE:   Generate and load ERP source tables into Azure SQL
# SOURCE:    Faker generated data
# TARGET:    Azure SQL Database (retailmax-erp)
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from faker import Faker
import random
import uuid



# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

# ── Retrieve Secrets ──────────────────────────────────────────
sql_server     = dbutils.secrets.get(scope=scope_name, key="sql-server")
sql_database = dbutils.secrets.get(scope=scope_name, key="sql-database")
sql_user     = dbutils.secrets.get(scope=scope_name, key="sql-user")
sql_password     = dbutils.secrets.get(scope=scope_name, key="sql-password")

jdbc_url = f'jdbc:sqlserver://{sql_server}:1433;database={sql_database};encrypt=true;trustServerCertificate=false; hostNameInCertificate=*.database.windows.net;loginTimeout=30;'

jdbc_properties = {
    "user": sql_user,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"  
}



In [0]:
# ============================================================
# Generate and Load ERP Source Data to Azure SQL
# ============================================================

from faker import Faker
from datetime import datetime, timedelta
import random

fake = Faker()

# ── Reference Data ─────────────────────────────────────────────
REGIONS = ["North", "South", "East", "West"]
STORE_TYPES = ["Mall", "High Street", "Outlet", "Airport"]
STATES = ["California", "Texas", "New York", "Florida", "Illinois", 
          "Ohio", "Georgia", "Washington", "Arizona", "Colorado"]

CATEGORIES = {
    "Apparel": ["Tops", "Bottoms", "Footwear", "Accessories"],
    "Electronics": ["Phones", "Laptops", "Audio", "Accessories"],
    "Home & Garden": ["Furniture", "Kitchen", "Decor", "Garden"],
    "Sports": ["Fitness", "Outdoor", "Team Sports", "Yoga"]
}

BRANDS = ["BrandX", "ProLine", "ValueMax", "PremiumCo", "EcoGoods", "TechBrand"]
SUPPLIERS = ["SupplierA", "GlobalGoods", "FastShip", "QualityFirst", "BulkMart"]

ORDER_STATUSES = ["Completed", "Pending", "Shipped", "Cancelled", "Returned"]
PAYMENT_METHODS = ["Credit Card", "Debit Card", "Cash", "Mobile Pay", "Gift Card"]
SALES_CHANNELS = ["In-Store", "Online", "Mobile App"]


# ── Generator Functions ────────────────────────────────────────

def generate_stores(n_stores=20):
    stores = []
    for i in range(1, n_stores + 1):
        stores.append({
            "store_id": i,
            "store_name": f"Store {fake.city()}",
            "city": fake.city(),
            "state": random.choice(STATES),
            "region": random.choice(REGIONS),
            "store_type": random.choice(STORE_TYPES),
            "open_date": fake.date_between(start_date="-10y", end_date="-1y"),
            "is_active": random.choice([True, True, True, False]),
            "updated_at": datetime.now()
        })
    return stores


def generate_products(n_products=500):
    products = []
    for i in range(1, n_products + 1):
        category = random.choice(list(CATEGORIES.keys()))
        subcategory = random.choice(CATEGORIES[category])
        products.append({
            "product_id": i,
            "product_name": fake.catch_phrase(),
            "category": category,
            "subcategory": subcategory,
            "brand": random.choice(BRANDS),
            "unit_price": round(random.uniform(5.0, 500.0), 2),
            "supplier_name": random.choice(SUPPLIERS),
            "is_active": random.choice([True, True, True, False]),
            "updated_at": datetime.now()
        })
    return products


def generate_sales_orders(n_orders=1000, n_stores=20, n_customers=2000):
    orders = []
    for i in range(1, n_orders + 1):
        order_ts = fake.date_time_between(start_date="-90d", end_date="now")
        orders.append({
            "order_id": i,
            "customer_id": random.randint(1, n_customers),
            "store_id": random.randint(1, n_stores),
            "order_ts": order_ts,
            "order_status": random.choice(ORDER_STATUSES),
            "payment_method": random.choice(PAYMENT_METHODS),
            "sales_channel": random.choice(SALES_CHANNELS),
            "total_amount": round(random.uniform(10.0, 2000.0), 2),
            "updated_at": datetime.now()
        })
    return orders


# ── Generate Data ──────────────────────────────────────────────
stores_data = generate_stores(n_stores=20)
products_data = generate_products(n_products=500)
orders_data = generate_sales_orders(n_orders=1000)

print(f"✅ Generated {len(stores_data)} stores")
print(f"✅ Generated {len(products_data)} products")
print(f"✅ Generated {len(orders_data)} orders")


# ── Convert to DataFrames ──────────────────────────────────────
df_stores = spark.createDataFrame(stores_data)
df_products = spark.createDataFrame(products_data)
df_orders = spark.createDataFrame(orders_data)

print("\n📋 Stores schema:")
df_stores.printSchema()

print("📋 Products schema:")
df_products.printSchema()

print("📋 Orders schema:")
df_orders.printSchema()

In [0]:
# ── Write DataFrames to Azure SQL ──────────────────────────────
# This notebook simulates ERP source system data
# Run once to set up source tables
# Do not rerun in production - append logic handled separately

# dim_stores
df_stores.write.jdbc(
    url=jdbc_url,
    table="dbo.dim_stores",
    mode="overwrite",
    properties=jdbc_properties
)
print("✅ dim_stores written to Azure SQL")

# dim_products
df_products.write.jdbc(
    url=jdbc_url,
    table="dbo.dim_products",
    mode="overwrite",
    properties=jdbc_properties
)
print("✅ dim_products written to Azure SQL")

# erp_sales_orders
df_orders.write.jdbc(
    url=jdbc_url,
    table="dbo.erp_sales_orders",
    mode="overwrite",
    properties=jdbc_properties
)
print("✅ erp_sales_orders written to Azure SQL")

print("\n🎉 All source tables loaded to Azure SQL!")

print("=" * 50)
print(f"dim_stores     : {len(stores_data)} rows")
print(f"dim_products   : {len(products_data)} rows")    
print(f"erp_sales_orders: {len(orders_data)} rows")
print("=" * 50)